# ANCP — D3 Exploratory Data Analysis
## IndiGo Profitability Intelligence System

**Primary target:** next-quarter PAT margin (%). **Secondary measure:** quarterly EBITDA margin (%).

This notebook provides a reproducible D3 EDA workflow. Financial outcomes are reconstructed from the raw Bloomberg-format `IS` sheet because the master workbook's legacy `Financial source` sheet is shifted by two quarters. Rows marked `Est` are excluded from actual history.

The project standard is Indian fiscal quarters: Q1 = Apr–Jun, Q2 = Jul–Sep, Q3 = Oct–Dec, and Q4 = Jan–Mar. Same-quarter market means here are for descriptive EDA only; predictive features need publication-aware cutoffs.

Keep source workbooks local and do not commit Bloomberg exports or output-rich reports to this public repository unless the applicable licence permits redistribution.

## 1. Business question and pre-specified hypotheses

**Question:** How do IndiGo's quarterly profitability margins vary alongside the available oil-price proxy, USD/INR, CPI and WPI indicators, and which variables merit further investigation for next-quarter PAT-margin forecasting?

These hypotheses are recorded before results are reviewed:

| ID | Hypothesis | Planned evidence |
|---|---|---|
| H1 | Oil benchmark is associated with PAT margin. | Spearman rho, paired n, p-value |
| H2 | USD/INR is associated with PAT margin. | Spearman rho, paired n, p-value |
| H3 | CPI inflation is associated with PAT margin. | Spearman rho, paired n, p-value |
| H4 | WPI inflation is associated with PAT margin. | Spearman rho, paired n, p-value |
| H5 | PAT margin differs across Indian fiscal quarters. | Kruskal–Wallis and quarter summaries |
| H6 | PAT and EBITDA margins move together. | Spearman rho, paired n, p-value |

Correlation is not causation. Small-sample tests and multiple comparisons require caution; unavailable relationships must be marked not assessable rather than imputed.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return Path(paths[0])

RAW_FILE = first_existing([
    ROOT / "data/raw/CP(4).xlsx", ROOT / "data/raw/CP.xlsx", ROOT / "data/raw/CP(3).xlsx",
    Path.cwd() / "CP(4).xlsx", Path.cwd() / "CP.xlsx", Path.cwd() / "CP(3).xlsx",
    Path("/mnt/data/CP(4).xlsx"), Path("/mnt/data/CP.xlsx"), Path("/mnt/data/CP(3).xlsx")
])
MASTER_FILE = first_existing([
    ROOT / "data/processed/CP Master Data(3).xlsx", ROOT / "data/processed/CP Master Data.xlsx",
    Path.cwd() / "CP Master Data(3).xlsx", Path.cwd() / "CP Master Data.xlsx", Path("/mnt/data/CP Master Data(3).xlsx"), Path("/mnt/data/CP Master Data.xlsx")
])
if not RAW_FILE.exists():
    raise FileNotFoundError("Place the latest raw Bloomberg export at data/raw/CP(4).xlsx.")
if not MASTER_FILE.exists():
    raise FileNotFoundError("Place CP Master Data(3).xlsx at data/processed/CP Master Data(3).xlsx.")

print("Raw financial source:", RAW_FILE)
print("Market-indicator master:", MASTER_FILE)

## 2. Load sources and audit the raw data

The raw `IS` sheet supplies Revenue, Net Income (GAAP) and EBITDA. Market series are taken from the corresponding sheets in the master workbook. The master workbook's `Financial source` sheet is audited for alignment but is not used as the target source.

In [ ]:
master_xls = pd.ExcelFile(MASTER_FILE)
master = {name: pd.read_excel(MASTER_FILE, sheet_name=name) for name in master_xls.sheet_names}
raw_is = pd.read_excel(RAW_FILE, sheet_name="IS", header=None)

# Validate the documented Bloomberg layout before selecting rows.
expected_rows = {5: "Revenue", 27: "Net Income, GAAP", 48: "EBITDA"}
for row_idx, expected in expected_rows.items():
    actual = str(raw_is.iloc[row_idx, 0]).strip()
    if actual != expected:
        raise ValueError(f"Unexpected IS layout at Excel row {row_idx+1}: {actual!r}; expected {expected!r}")

fin_all = pd.DataFrame({
    "Date": pd.to_datetime(raw_is.iloc[4, 2:], errors="coerce").to_numpy(),
    "Bloomberg quarter header": raw_is.iloc[3, 2:].astype(str).to_numpy(),
    "Revenue": pd.to_numeric(raw_is.iloc[5, 2:], errors="coerce").to_numpy(),
    "PAT": pd.to_numeric(raw_is.iloc[27, 2:], errors="coerce").to_numpy(),
    "EBITDA": pd.to_numeric(raw_is.iloc[48, 2:], errors="coerce").to_numpy(),
})
fin_all["is_estimate"] = fin_all["Bloomberg quarter header"].str.contains(r"\bEst\b", case=False, na=False)
fin_all["PAT Margin"] = np.where(fin_all["Revenue"] > 0, fin_all["PAT"] / fin_all["Revenue"] * 100, np.nan)
fin_all["EBITDA Margin"] = np.where(fin_all["Revenue"] > 0, fin_all["EBITDA"] / fin_all["Revenue"] * 100, np.nan)

display(pd.DataFrame([{
    "source": "Raw Bloomberg IS",
    "rows_in_export": len(fin_all),
    "valid_dates": int(fin_all["Date"].notna().sum()),
    "actual_target_rows": int((~fin_all["is_estimate"] & fin_all["PAT Margin"].notna()).sum()),
    "estimated_rows_excluded": int(fin_all["is_estimate"].sum()),
    "first_actual_date": fin_all.loc[~fin_all["is_estimate"], "Date"].min(),
    "last_actual_date": fin_all.loc[~fin_all["is_estimate"], "Date"].max(),
}]))

market_sheet_names = [
    "USD INR", "CPI inflation", "WPI inflation",
    "Oil benchmark(MONTHLY)", "Oil benchmark(DAILY)"
]
audit = []
for name, frame in master.items():
    if {"Date", "PX_LAST"}.issubset(frame.columns):
        dates = pd.to_datetime(frame["Date"], errors="coerce")
        values = pd.to_numeric(frame["PX_LAST"], errors="coerce")
        audit.append({
            "sheet": name, "rows": len(frame), "unique_dates": int(dates.nunique()),
            "first_date": dates.min(), "last_date": dates.max(),
            "missing_or_invalid_date_value": int(dates.isna().sum() + values.isna().sum())
        })
    else:
        audit.append({"sheet": name, "rows": len(frame), "unique_dates": np.nan,
                      "first_date": None, "last_date": None,
                      "missing_or_invalid_date_value": int(frame.isna().sum().sum())})
display(pd.DataFrame(audit))

## 3. Reconcile the legacy master financial sheet

The D2 audit flagged a two-quarter shift in the master workbook's legacy financial sheet. This cell compares the same-date values with the values moved forward by two calendar quarters. The legacy sheet is excluded from target construction regardless of this diagnostic.

In [ ]:
legacy = master.get("Financial source")
if legacy is not None and {"Date", "PAT Margin", "EBITDA Margin"}.issubset(legacy.columns):
    m = legacy[["Date", "PAT Margin", "EBITDA Margin"]].copy()
    m["Date"] = pd.to_datetime(m["Date"], errors="coerce")
    for col in ["PAT Margin", "EBITDA Margin"]:
        m[col] = pd.to_numeric(m[col], errors="coerce")

    raw_compare = fin_all[["Date", "PAT Margin", "EBITDA Margin"]].dropna(subset=["Date"]).copy()
    raw_compare["date_plus_two_quarters"] = (raw_compare["Date"].dt.to_period("Q") + 2).dt.end_time.dt.normalize()

    same_date = m.merge(raw_compare, on="Date", how="inner", suffixes=("_master", "_raw"))
    shifted = m.merge(raw_compare, left_on="Date", right_on="date_plus_two_quarters",
                      how="inner", suffixes=("_master", "_raw"))

    def match_pct(frame):
        if frame.empty:
            return np.nan
        a = np.isclose(frame["PAT Margin_master"], frame["PAT Margin_raw"], atol=0.02, rtol=0)
        b = np.isclose(frame["EBITDA Margin_master"], frame["EBITDA Margin_raw"], atol=0.02, rtol=0)
        return float((a & b).mean() * 100)

    display(pd.DataFrame([
        {"comparison": "Same quarter-end date", "overlap_rows": len(same_date), "both margins match (%)": match_pct(same_date)},
        {"comparison": "Raw values moved forward two quarters", "overlap_rows": len(shifted), "both margins match (%)": match_pct(shifted)},
    ]))
else:
    print("Legacy Financial source sheet not available in the expected format.")

## 4. Clean inputs and build the fiscal-quarter panel

Target observations are taken only from actual (non-estimated) financial periods. Market indicators are grouped by fiscal quarter using `Q-MAR`. A quarter lacking a market value remains missing; it is not silently filled.

In [ ]:
fin = fin_all.loc[~fin_all["is_estimate"]].dropna(subset=["Date", "Revenue", "PAT"]).copy()
fin = fin.loc[fin["Revenue"] > 0].copy()
fin["quarter_end"] = fin["Date"].dt.to_period("Q-MAR").dt.end_time.dt.normalize()
fin = fin.sort_values("quarter_end").drop_duplicates("quarter_end", keep=False)

market_map = {
    "USD INR": "usd_inr_mean",
    "CPI inflation": "cpi_inflation_mean",
    "WPI inflation": "wpi_inflation_mean",
    "Oil benchmark(MONTHLY)": "oil_benchmark_monthly_mean",
    "Oil benchmark(DAILY)": "oil_benchmark_daily_mean",
}
quarterly_market = []
for sheet_name, output_name in market_map.items():
    if sheet_name not in master:
        continue
    x = master[sheet_name][["Date", "PX_LAST"]].copy()
    x["Date"] = pd.to_datetime(x["Date"], errors="coerce")
    x["PX_LAST"] = pd.to_numeric(x["PX_LAST"], errors="coerce")
    x = x.dropna(subset=["Date", "PX_LAST"])
    x["quarter_end"] = x["Date"].dt.to_period("Q-MAR").dt.end_time.dt.normalize()
    quarterly_market.append(
        x.groupby("quarter_end", as_index=False)["PX_LAST"].mean().rename(columns={"PX_LAST": output_name})
    )

panel = fin.copy()
for qdf in quarterly_market:
    panel = panel.merge(qdf, on="quarter_end", how="left", validate="one_to_one")

period = panel["quarter_end"].dt.to_period("Q-MAR")
panel["fiscal_quarter"] = "Q" + period.dt.quarter.astype(str)
fy_end = period.dt.qyear
panel["fiscal_year"] = "FY" + (fy_end - 1).astype(str) + "-" + fy_end.astype(str).str[-2:]
panel["quarter_label"] = panel["fiscal_year"] + "_" + panel["fiscal_quarter"]
panel = panel.sort_values("quarter_end").reset_index(drop=True)

display(panel[["quarter_label", "quarter_end", "Revenue", "PAT", "EBITDA", "PAT Margin", "EBITDA Margin"]].head())
display(panel[["quarter_label", "quarter_end", "PAT Margin", "EBITDA Margin"]].tail())

## 5. Repeat data-quality and representativeness checks after cleaning

In [ ]:
market_cols = [c for c in panel.columns if c.endswith("_mean")]
quality = pd.DataFrame([{
    "rows": len(panel),
    "unique_quarter_ends": int(panel["quarter_end"].nunique()),
    "first_quarter_end": panel["quarter_end"].min(),
    "last_quarter_end": panel["quarter_end"].max(),
    "PAT_margin_nonmissing": int(panel["PAT Margin"].notna().sum()),
    "EBITDA_margin_nonmissing": int(panel["EBITDA Margin"].notna().sum()),
    "all_market_inputs_present": int(panel[market_cols].notna().all(axis=1).sum()) if market_cols else 0,
    "duplicate_quarter_ends": int(panel["quarter_end"].duplicated().sum()),
}])
display(quality)

missingness = pd.DataFrame({
    "column": panel.columns,
    "dtype": [str(panel[c].dtype) for c in panel.columns],
    "missing_n": [int(panel[c].isna().sum()) for c in panel.columns],
    "missing_pct": [round(float(panel[c].isna().mean() * 100), 2) for c in panel.columns],
})
display(missingness)
print("Representativeness: this is one airline observed over time, not a random sample of airlines.")

## 6. Distributions, trends and segments

In [ ]:
numeric_cols = ["PAT Margin", "EBITDA Margin"] + market_cols
display(panel[numeric_cols].describe().T)

for col in ["PAT Margin", "EBITDA Margin"]:
    values = panel[col].dropna()
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.hist(values, bins="auto", edgecolor="white")
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel("Margin (%)")
    ax.set_ylabel("Quarter count")
    plt.tight_layout()
    plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(panel["quarter_end"], panel["PAT Margin"], marker="o", label="PAT margin (%)")
ax.plot(panel["quarter_end"], panel["EBITDA Margin"], marker="o", label="EBITDA margin (%)")
ax.set_title("IndiGo quarterly profitability margins")
ax.set_xlabel("Fiscal quarter end")
ax.set_ylabel("Margin (%)")
ax.legend()
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

for col in ["usd_inr_mean", "cpi_inflation_mean", "wpi_inflation_mean", "oil_benchmark_monthly_mean"]:
    if col in panel:
        fig, ax = plt.subplots(figsize=(9, 3.5))
        ax.scatter(panel[col], panel["PAT Margin"], alpha=0.8)
        ax.set_title(f"PAT margin vs {col.replace('_', ' ')} (exploratory)")
        ax.set_xlabel(col.replace("_", " "))
        ax.set_ylabel("PAT margin (%)")
        plt.tight_layout()
        plt.show()

quarter_summary = panel.groupby("fiscal_quarter")["PAT Margin"].agg(["count", "mean", "median", "std", "min", "max"]).reindex(["Q1", "Q2", "Q3", "Q4"])
display(quarter_summary)
fig, ax = plt.subplots(figsize=(8, 4))
data = [panel.loc[panel["fiscal_quarter"] == q, "PAT Margin"].dropna().to_numpy() for q in ["Q1", "Q2", "Q3", "Q4"]]
ax.boxplot(data, tick_labels=["Q1 Apr–Jun", "Q2 Jul–Sep", "Q3 Oct–Dec", "Q4 Jan–Mar"], showfliers=True)
ax.set_title("PAT margin by Indian fiscal quarter")
ax.set_ylabel("PAT margin (%)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 7. Hypothesis testing and insight log

Tests are exploratory. Report paired observation counts and effect sizes with p-values. Do not interpret statistical significance as evidence of a causal or predictive relationship.

In [ ]:
hypotheses = [
    ("H1", "Oil benchmark and PAT margin are associated", "oil_benchmark_monthly_mean", "PAT Margin"),
    ("H2", "USD/INR and PAT margin are associated", "usd_inr_mean", "PAT Margin"),
    ("H3", "CPI inflation and PAT margin are associated", "cpi_inflation_mean", "PAT Margin"),
    ("H4", "WPI inflation and PAT margin are associated", "wpi_inflation_mean", "PAT Margin"),
    ("H6", "PAT and EBITDA margins move together", "PAT Margin", "EBITDA Margin"),
]
results = []
for hid, statement, xcol, ycol in hypotheses:
    sub = panel[[xcol, ycol]].dropna() if xcol in panel else pd.DataFrame()
    if len(sub) < 5 or sub[xcol].nunique() < 2 or sub[ycol].nunique() < 2:
        results.append({"ID": hid, "hypothesis": statement, "n": len(sub), "rho": np.nan, "p_value": np.nan, "assessment": "Not assessable"})
    else:
        rho, p = stats.spearmanr(sub[xcol], sub[ycol])
        results.append({"ID": hid, "hypothesis": statement, "n": len(sub), "rho": rho, "p_value": p,
                        "assessment": "Significant at 5%" if p < 0.05 else "Not significant at 5%"})
hyp_results = pd.DataFrame(results)

groups = [g["PAT Margin"].dropna().to_numpy() for _, g in panel.groupby("fiscal_quarter") if g["PAT Margin"].notna().sum() >= 2]
if len(groups) >= 2:
    hstat, hp = stats.kruskal(*groups)
    h5 = {"ID": "H5", "hypothesis": "PAT margin differs across fiscal quarters", "n": sum(len(g) for g in groups),
          "rho": np.nan, "p_value": hp, "test_statistic": hstat,
          "assessment": "Significant at 5%" if hp < 0.05 else "Not significant at 5%"}
else:
    h5 = {"ID": "H5", "hypothesis": "PAT margin differs across fiscal quarters", "n": sum(len(g) for g in groups),
          "rho": np.nan, "p_value": np.nan, "test_statistic": np.nan, "assessment": "Not assessable"}
hyp_results = pd.concat([hyp_results, pd.DataFrame([h5])], ignore_index=True)
display(hyp_results)

insights = [{
    "Insight": "PAT margin varies across the observed period.",
    "Evidence": f"n={panel['PAT Margin'].notna().sum()}; range {panel['PAT Margin'].min():.2f}% to {panel['PAT Margin'].max():.2f}%.",
    "So what": "One average may not describe every quarter.",
    "Modelling implication": "Inspect unusual quarters and evaluate chronologically.",
    "Confidence": "High for description; low for explanation."
}]
for _, row in hyp_results.iterrows():
    if pd.notna(row.get("rho")) and pd.notna(row.get("p_value")):
        ev = f"n={int(row['n'])}; Spearman rho={row['rho']:.2f}; p={row['p_value']:.3f}."
    elif row["ID"] == "H5" and pd.notna(row.get("test_statistic")) and pd.notna(row.get("p_value")):
        ev = f"n={int(row['n'])}; Kruskal-Wallis H={row['test_statistic']:.2f}; p={row['p_value']:.3f}."
    else:
        ev = f"n={int(row['n'])}; not assessable with the available paired data."
    insights.append({
        "Insight": row["hypothesis"], "Evidence": ev,
        "So what": "Treat associations as exploratory; correlation is not causation.",
        "Modelling implication": "Retain a feature only if it improves time-based out-of-sample results.",
        "Confidence": "Low-to-medium for generalisation; the sample is small and time-dependent."
    })
insights.append({
    "Insight": "The dataset is a single-airline time series.",
    "Evidence": f"{len(panel)} actual quarterly target rows after excluding estimates.",
    "So what": "Industry-wide conclusions are not supported.",
    "Modelling implication": "Start with a simple baseline and parsimonious candidates.",
    "Confidence": "High for this sample-design limitation."
})
insight_log = pd.DataFrame(insights)
display(insight_log)

## 8. Limitations and next steps

- The target dataset is small and covers one airline; complex models may overfit.
- The master workbook's legacy Financial source sheet is shifted by two quarters and is not used as a target.
- Bloomberg periods marked Est are not actual results and are excluded from the observed history.
- The supplied flat EDA analysis workbook repeats margins in the first two quarter rows while a separate dataset marks those periods as unavailable; rebuild it from raw financials.
- The oil benchmark is not confirmed as Indian ATF or IndiGo's realised fuel cost.
- The CPI series has a date irregularity and incomplete coverage in at least one quarter; verify the source and retain missingness rather than inventing values.
- Same-quarter market means are descriptive EDA only. Model inputs must be available at the relevant forecast cutoff.
- No causal claim and no model-performance claim are made here.

**Next:** rebuild the final quarterly dataset and calculation audit from the raw IS sheet, verify official financial line items and publication dates, rerun the notebook locally, then proceed to the D4 baseline.